# Deep Learning WCE Classification Project

**Objective:** Build and compare deep learning models to classify GI diseases from endoscopy images, handling class imbalance and comparing against a naturally balanced dataset.

**Datasets:**
- [Kvasir-Capsule](https://osf.io/dv2ag/) — imbalanced (14 classes, ~47K images)
- [KVASIR v2](https://datasets.simula.no/kvasir/) — balanced (8 classes, 1000 each)

**Models:** EfficientNetB0, MobileNetV2, ResNet101V2

---

---
## 🚀 Google Colab Setup (skip if running locally)

The cell below auto-detects whether you are running on **Google Colab**.  
If so it will:
1. Clone / mount the project repo so the `src/` modules are importable.
2. Install extra Python dependencies (`uv` is not available on Colab, so `pip` is used).
3. Download both datasets – choose **one** of the three strategies that fits your situation:

| Strategy | What to do |
|----------|------------|
| **A – Kaggle API** | Upload your `kaggle.json` key; the cell installs `kaggle` and pulls both datasets automatically. |
| **B – Google Drive** | Upload the zip files to your Drive, mount Drive, then set the `DRIVE_ZIP_*` paths below. |
| **C – Direct URL** | Provide direct-download URLs for the zip files in the `DIRECT_URL_*` variables below. |

> **Local users:** This cell is a no-op – nothing is installed or downloaded.


In [ ]:
# ---- Google Colab dataset bootstrap -----------------------------------------
# This cell is a no-op when executed outside of Google Colab.

import sys, os

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ---- 1. Clone repo (if not already present) ------------------------------
    REPO_URL = "https://github.com/YOUR_USERNAME/DLMinorProject.git"  # <-- UPDATE
    REPO_DIR = "/content/DLMinorProject"

    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    print(f"Working directory: {os.getcwd()}")

    # ---- 2. Install Python dependencies -------------------------------------
    os.system("pip install -q tensorflow keras pillow seaborn scikit-learn matplotlib")

    # ---- 3. Dataset paths ---------------------------------------------------
    DATASET_ROOT   = "/content/DLMinorProject/data"
    CAPSULE_DEST   = os.path.join(DATASET_ROOT, "labeled-images")
    KVASIR_V2_DEST = os.path.join(DATASET_ROOT, "kvasir-dataset-v2")

    os.makedirs(CAPSULE_DEST,   exist_ok=True)
    os.makedirs(KVASIR_V2_DEST, exist_ok=True)

    # ---- Helper: smart extract ----------------------------------------------
    # Strips the zip's single top-level folder (if present) so that class
    # sub-folders land directly inside `dest_dir` instead of being double-nested.
    def smart_extract(zip_path, dest_dir):
        import zipfile, shutil
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            # Detect common top-level prefix (e.g. "kvasir-dataset-v2/")
            top_dirs = {n.split("/")[0] for n in names if "/" in n}
            # If every entry shares one prefix, strip it during extraction
            if len(top_dirs) == 1:
                prefix = top_dirs.pop() + "/"
                tmp_dir = dest_dir + "_tmp_extract"
                os.makedirs(tmp_dir, exist_ok=True)
                zf.extractall(tmp_dir)
                src = os.path.join(tmp_dir, prefix.rstrip("/"))
                # Move contents of src into dest_dir
                for item in os.listdir(src):
                    s = os.path.join(src, item)
                    d = os.path.join(dest_dir, item)
                    if os.path.exists(d):
                        if os.path.isdir(d):
                            shutil.rmtree(d)
                        else:
                            os.remove(d)
                    shutil.move(s, d)
                shutil.rmtree(tmp_dir)
                print(f"Extracted (stripped prefix '{prefix}') -> {dest_dir}")
            else:
                zf.extractall(dest_dir)
                print(f"Extracted -> {dest_dir}")

    # ---- Strategy A: Kaggle API ---------------------------------------------
    USE_KAGGLE = False   # <-- set True to use Kaggle API
    if USE_KAGGLE:
        from google.colab import files
        print("Upload your kaggle.json ...")
        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.system("cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json")
        os.system("pip install -q kaggle")
        # Downloads zip files then we extract with smart_extract below
        os.system(f"kaggle datasets download -d meladsamuel/kvasir-capsule -p {DATASET_ROOT}")
        os.system(f"kaggle datasets download -d meladsamuel/kvasir -p {DATASET_ROOT}")
        # Find the downloaded zips and extract
        for zname, dest in [("kvasir-capsule.zip", CAPSULE_DEST),
                             ("kvasir.zip",         KVASIR_V2_DEST)]:
            zp = os.path.join(DATASET_ROOT, zname)
            if os.path.isfile(zp):
                smart_extract(zp, dest)
                os.remove(zp)

    # ---- Strategy B: Google Drive -------------------------------------------
    USE_DRIVE = False    # <-- set True to pull from Google Drive
    DRIVE_ZIP_CAPSULE  = "/content/drive/MyDrive/datasets/kvasir-capsule.zip"    # <-- your path
    DRIVE_ZIP_V2       = "/content/drive/MyDrive/datasets/kvasir-dataset-v2.zip" # <-- your path
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isfile(DRIVE_ZIP_CAPSULE):
            smart_extract(DRIVE_ZIP_CAPSULE, CAPSULE_DEST)
        if os.path.isfile(DRIVE_ZIP_V2):
            smart_extract(DRIVE_ZIP_V2, KVASIR_V2_DEST)

    # ---- Strategy C: Direct URL download ------------------------------------
    USE_DIRECT_URL = False   # <-- set True to download from direct URLs
    DIRECT_URL_CAPSULE = "https://REPLACE_WITH_DIRECT_LINK_TO_kvasir-capsule.zip"
    DIRECT_URL_V2      = "https://REPLACE_WITH_DIRECT_LINK_TO_kvasir-dataset-v2.zip"
    if USE_DIRECT_URL:
        import urllib.request
        for url, dest in [(DIRECT_URL_CAPSULE, CAPSULE_DEST),
                          (DIRECT_URL_V2,      KVASIR_V2_DEST)]:
            zip_path = dest + ".zip"
            print(f"Downloading {url} ...")
            urllib.request.urlretrieve(url, zip_path)
            smart_extract(zip_path, dest)
            os.remove(zip_path)

    # ---- 4. Patch DATA_DIR / KVASIR_V2_DIR ----------------------------------
    import src.data_utils as _du
    from pathlib import Path
    _du.DATA_DIR      = Path(CAPSULE_DEST)
    _du.KVASIR_V2_DIR = Path(KVASIR_V2_DEST)

    print("\n✅ Colab setup complete.")
    print(f"   Kvasir-Capsule : {_du.DATA_DIR}")
    print(f"   KVASIR v2      : {_du.KVASIR_V2_DIR}")
else:
    print("Running locally – no dataset download needed.")

## Setup

---
## Training — One Cell per Scenario × Model

Each cell trains **one** model for **one** scenario, then calls `clear_session()` to free GPU/CPU memory before the next run.


In [ ]:
# Accumulator for all evaluation results
all_results = {}


---
## S1 — Baseline (raw Kvasir-Capsule, imbalanced)


---
## Training — One Cell per Scenario × Model

Each cell trains **one** model for **one** scenario, then calls `clear_session()` to free RAM before the next run.


In [ ]:
# Accumulator  →  {setting: {arch: metrics_dict}}
all_results = {}


---
## S1 — Baseline (raw Kvasir-Capsule, imbalanced)


In [ ]:
# ── S1 — Baseline (raw Kvasir-Capsule, imbalanced)  |  EfficientNetB0 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(capsule_images)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('EfficientNetB0', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s1_baseline/efficientnetb0').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s1_baseline_efficientnetb0',
)

# 4. Plot history
plot_training_history(
    history,
    title='S1 — Baseline (raw Kvasir-Capsule, imbalanced) – EfficientNetB0',
    save_path='../outputs/s1_baseline/efficientnetb0/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S1 — Baseline (raw Kvasir-Capsule, imbalanced)'] = all_results.get('S1 — Baseline (raw Kvasir-Capsule, imbalanced)', {})
all_results['S1 — Baseline (raw Kvasir-Capsule, imbalanced)']['EfficientNetB0'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S1 — Baseline (raw Kvasir-Capsule, imbalanced) – EfficientNetB0',
    save_path='../outputs/s1_baseline/efficientnetb0/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


In [ ]:
# ── S1 — Baseline (raw Kvasir-Capsule, imbalanced)  |  MobileNetV2 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(capsule_images)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('MobileNetV2', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s1_baseline/mobilenetv2').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s1_baseline_mobilenetv2',
)

# 4. Plot history
plot_training_history(
    history,
    title='S1 — Baseline (raw Kvasir-Capsule, imbalanced) – MobileNetV2',
    save_path='../outputs/s1_baseline/mobilenetv2/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S1 — Baseline (raw Kvasir-Capsule, imbalanced)'] = all_results.get('S1 — Baseline (raw Kvasir-Capsule, imbalanced)', {})
all_results['S1 — Baseline (raw Kvasir-Capsule, imbalanced)']['MobileNetV2'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S1 — Baseline (raw Kvasir-Capsule, imbalanced) – MobileNetV2',
    save_path='../outputs/s1_baseline/mobilenetv2/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


In [ ]:
# ── S1 — Baseline (raw Kvasir-Capsule, imbalanced)  |  ResNet101V2 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(capsule_images)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('ResNet101V2', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s1_baseline/resnet101v2').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s1_baseline_resnet101v2',
)

# 4. Plot history
plot_training_history(
    history,
    title='S1 — Baseline (raw Kvasir-Capsule, imbalanced) – ResNet101V2',
    save_path='../outputs/s1_baseline/resnet101v2/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S1 — Baseline (raw Kvasir-Capsule, imbalanced)'] = all_results.get('S1 — Baseline (raw Kvasir-Capsule, imbalanced)', {})
all_results['S1 — Baseline (raw Kvasir-Capsule, imbalanced)']['ResNet101V2'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S1 — Baseline (raw Kvasir-Capsule, imbalanced) – ResNet101V2',
    save_path='../outputs/s1_baseline/resnet101v2/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


---
## S2 — Under-Sampling only (Kvasir-Capsule)


In [ ]:
# ── S2 — Under-Sampling only (Kvasir-Capsule)  |  EfficientNetB0 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(undersampled)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('EfficientNetB0', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s2_undersample/efficientnetb0').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s2_undersample_efficientnetb0',
)

# 4. Plot history
plot_training_history(
    history,
    title='S2 — Under-Sampling only (Kvasir-Capsule) – EfficientNetB0',
    save_path='../outputs/s2_undersample/efficientnetb0/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S2 — Under-Sampling only (Kvasir-Capsule)'] = all_results.get('S2 — Under-Sampling only (Kvasir-Capsule)', {})
all_results['S2 — Under-Sampling only (Kvasir-Capsule)']['EfficientNetB0'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S2 — Under-Sampling only (Kvasir-Capsule) – EfficientNetB0',
    save_path='../outputs/s2_undersample/efficientnetb0/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


In [ ]:
# ── S2 — Under-Sampling only (Kvasir-Capsule)  |  MobileNetV2 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(undersampled)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('MobileNetV2', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s2_undersample/mobilenetv2').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s2_undersample_mobilenetv2',
)

# 4. Plot history
plot_training_history(
    history,
    title='S2 — Under-Sampling only (Kvasir-Capsule) – MobileNetV2',
    save_path='../outputs/s2_undersample/mobilenetv2/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S2 — Under-Sampling only (Kvasir-Capsule)'] = all_results.get('S2 — Under-Sampling only (Kvasir-Capsule)', {})
all_results['S2 — Under-Sampling only (Kvasir-Capsule)']['MobileNetV2'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S2 — Under-Sampling only (Kvasir-Capsule) – MobileNetV2',
    save_path='../outputs/s2_undersample/mobilenetv2/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


In [ ]:
# ── S2 — Under-Sampling only (Kvasir-Capsule)  |  ResNet101V2 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(undersampled)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('ResNet101V2', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s2_undersample/resnet101v2').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s2_undersample_resnet101v2',
)

# 4. Plot history
plot_training_history(
    history,
    title='S2 — Under-Sampling only (Kvasir-Capsule) – ResNet101V2',
    save_path='../outputs/s2_undersample/resnet101v2/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S2 — Under-Sampling only (Kvasir-Capsule)'] = all_results.get('S2 — Under-Sampling only (Kvasir-Capsule)', {})
all_results['S2 — Under-Sampling only (Kvasir-Capsule)']['ResNet101V2'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S2 — Under-Sampling only (Kvasir-Capsule) – ResNet101V2',
    save_path='../outputs/s2_undersample/resnet101v2/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


---
## S3 — Under-Sampling + Augmentation (Kvasir-Capsule)


In [ ]:
# ── S3 — Under-Sampling + Augmentation (Kvasir-Capsule)  |  EfficientNetB0 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(balanced)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('EfficientNetB0', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s3_augmented/efficientnetb0').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s3_augmented_efficientnetb0',
)

# 4. Plot history
plot_training_history(
    history,
    title='S3 — Under-Sampling + Augmentation (Kvasir-Capsule) – EfficientNetB0',
    save_path='../outputs/s3_augmented/efficientnetb0/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S3 — Under-Sampling + Augmentation (Kvasir-Capsule)'] = all_results.get('S3 — Under-Sampling + Augmentation (Kvasir-Capsule)', {})
all_results['S3 — Under-Sampling + Augmentation (Kvasir-Capsule)']['EfficientNetB0'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S3 — Under-Sampling + Augmentation (Kvasir-Capsule) – EfficientNetB0',
    save_path='../outputs/s3_augmented/efficientnetb0/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


In [ ]:
# ── S3 — Under-Sampling + Augmentation (Kvasir-Capsule)  |  MobileNetV2 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(balanced)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('MobileNetV2', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s3_augmented/mobilenetv2').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s3_augmented_mobilenetv2',
)

# 4. Plot history
plot_training_history(
    history,
    title='S3 — Under-Sampling + Augmentation (Kvasir-Capsule) – MobileNetV2',
    save_path='../outputs/s3_augmented/mobilenetv2/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S3 — Under-Sampling + Augmentation (Kvasir-Capsule)'] = all_results.get('S3 — Under-Sampling + Augmentation (Kvasir-Capsule)', {})
all_results['S3 — Under-Sampling + Augmentation (Kvasir-Capsule)']['MobileNetV2'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S3 — Under-Sampling + Augmentation (Kvasir-Capsule) – MobileNetV2',
    save_path='../outputs/s3_augmented/mobilenetv2/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


In [ ]:
# ── S3 — Under-Sampling + Augmentation (Kvasir-Capsule)  |  ResNet101V2 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(balanced)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('ResNet101V2', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s3_augmented/resnet101v2').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s3_augmented_resnet101v2',
)

# 4. Plot history
plot_training_history(
    history,
    title='S3 — Under-Sampling + Augmentation (Kvasir-Capsule) – ResNet101V2',
    save_path='../outputs/s3_augmented/resnet101v2/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S3 — Under-Sampling + Augmentation (Kvasir-Capsule)'] = all_results.get('S3 — Under-Sampling + Augmentation (Kvasir-Capsule)', {})
all_results['S3 — Under-Sampling + Augmentation (Kvasir-Capsule)']['ResNet101V2'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S3 — Under-Sampling + Augmentation (Kvasir-Capsule) – ResNet101V2',
    save_path='../outputs/s3_augmented/resnet101v2/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


---
## S4 — Naturally Balanced (KVASIR v2)


In [ ]:
# ── S4 — Naturally Balanced (KVASIR v2)  |  EfficientNetB0 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(kvasir_v2_images)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('EfficientNetB0', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s4_kvasir_v2/efficientnetb0').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s4_kvasir_v2_efficientnetb0',
)

# 4. Plot history
plot_training_history(
    history,
    title='S4 — Naturally Balanced (KVASIR v2) – EfficientNetB0',
    save_path='../outputs/s4_kvasir_v2/efficientnetb0/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S4 — Naturally Balanced (KVASIR v2)'] = all_results.get('S4 — Naturally Balanced (KVASIR v2)', {})
all_results['S4 — Naturally Balanced (KVASIR v2)']['EfficientNetB0'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S4 — Naturally Balanced (KVASIR v2) – EfficientNetB0',
    save_path='../outputs/s4_kvasir_v2/efficientnetb0/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


In [ ]:
# ── S4 — Naturally Balanced (KVASIR v2)  |  MobileNetV2 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(kvasir_v2_images)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('MobileNetV2', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s4_kvasir_v2/mobilenetv2').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s4_kvasir_v2_mobilenetv2',
)

# 4. Plot history
plot_training_history(
    history,
    title='S4 — Naturally Balanced (KVASIR v2) – MobileNetV2',
    save_path='../outputs/s4_kvasir_v2/mobilenetv2/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S4 — Naturally Balanced (KVASIR v2)'] = all_results.get('S4 — Naturally Balanced (KVASIR v2)', {})
all_results['S4 — Naturally Balanced (KVASIR v2)']['MobileNetV2'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S4 — Naturally Balanced (KVASIR v2) – MobileNetV2',
    save_path='../outputs/s4_kvasir_v2/mobilenetv2/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


In [ ]:
# ── S4 — Naturally Balanced (KVASIR v2)  |  ResNet101V2 ──
from src.training import clear_session

# 1. Split data
splits = preprocess_dataset(kvasir_v2_images)
X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']
class_names      = splits['class_names']

# 2. Build & compile
model = build_model('ResNet101V2', num_classes=len(class_names))
compile_model(model)

# 3. Train
import pathlib
pathlib.Path('../outputs/s4_kvasir_v2/resnet101v2').mkdir(parents=True, exist_ok=True)
history = train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    batch_size=32,
    epochs=30,
    save_name='s4_kvasir_v2_resnet101v2',
)

# 4. Plot history
plot_training_history(
    history,
    title='S4 — Naturally Balanced (KVASIR v2) – ResNet101V2',
    save_path='../outputs/s4_kvasir_v2/resnet101v2/history.png',
)

# 5. Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)
all_results['S4 — Naturally Balanced (KVASIR v2)'] = all_results.get('S4 — Naturally Balanced (KVASIR v2)', {})
all_results['S4 — Naturally Balanced (KVASIR v2)']['ResNet101V2'] = metrics

# 6. Confusion matrix
plot_confusion_matrix(
    y_test, metrics['y_pred'], class_names,
    title='S4 — Naturally Balanced (KVASIR v2) – ResNet101V2',
    save_path='../outputs/s4_kvasir_v2/resnet101v2/confusion.png',
)

# 7. Free RAM
del model, history, splits, X_train, X_val, X_test, y_train, y_val, y_test
clear_session()


---
## Comparison


In [ ]:
# ── Final comparison across all scenarios & models ──
# all_results shape: {setting: {arch: metrics_dict}} — matches build_comparison_table
display_comparison(build_comparison_table(all_results))
